In [ ]:
# ==============================================================================
# 1) ENVIRONMENT — Kaggle có sẵn torch + transformers, chỉ cần thêm decord
# ==============================================================================
import sys
import importlib
import subprocess

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

ensure("decord")  # Đọc video nhanh; nếu cài lỗi, Cell 4 tự fallback sang OpenCV

import torch
import transformers
from packaging import version

assert version.parse(transformers.__version__) >= version.parse("4.36"), \
    f"Cần transformers>=4.36 (gradient_checkpointing_kwargs), hiện tại {transformers.__version__}"

try:
    import decord
    decord_info = decord.__version__
except ImportError:
    decord_info = "N/A (dùng OpenCV fallback)"

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs {torch.cuda.device_count()}")
print(f"Transformers {transformers.__version__} | decord {decord_info}")

In [ ]:
# ==============================================================================
# 2) CẤU HÌNH HỆ THỐNG & HYPERPARAMETERS
# ==============================================================================
from pathlib import Path
import torch

# ── Paths ──
PROJECT_ROOT = Path("/kaggle/working/school_violence_project")
RAW_DATA_ROOT = Path("/kaggle/input/datasets/doannh/dataset")
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "timesformer_hf_run"
BEST_MODEL_DIR = PROJECT_ROOT / "checkpoints" / "timesformer_best"
REPORT_DIR = PROJECT_ROOT / "reports"
SPLIT_CSV = PROJECT_ROOT / "splits.csv"

for p in [CHECKPOINT_DIR, BEST_MODEL_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ── TimeSformer Model Config ──
MODEL_NAME = "facebook/timesformer-base-finetuned-k400"
NUM_FRAMES = 16            # Model gốc 8 frames → Cell 4 nội suy time embedding lên 16
IMG_SIZE = 224
TRAIN_SCALE_RANGE = (256, 320)  # ĐÃ SỬA: (224,288) → (256,320) — đảm bảo luôn có spatial jitter thực sự
EVAL_RESIZE = 224               # Giống preprocessing gốc của TimeSformer: resize 224 → center crop 224

# ── Training Hyperparameters (2× T4 16GB) ──
NUM_GPUS = max(1, torch.cuda.device_count())
TRAIN_BATCH_SIZE = 2       # Per GPU — 16 frames × 196 patches = 3136 tokens/clip
EVAL_BATCH_SIZE = 4        # Eval không cần gradient → tăng batch để inference nhanh hơn
GRAD_ACCUM_STEPS = 4       # Effective batch = 2 × 4 × 2 GPU = 16
NUM_EPOCHS = 30            # ĐÃ SỬA: 16 → 30 epochs, cho model thêm headroom converge
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.1
LABEL_SMOOTHING = 0.1
EARLY_STOPPING_PATIENCE = 3  # MỚI: dừng nếu val f1_macro không cải thiện sau 3 epochs
NUM_WORKERS = 2
SEED = 42
GRADIENT_CHECKPOINTING = True   # Bắt buộc với 16 frames trên T4

# ── Data Split ──
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

# ── Class mapping — PHẢI match chính xác tên folder trên disk ──
CLASS_TO_IDX = {
    "Punching": 0, "Kicking": 1, "Slapping": 2, "Pushing": 3,
    "Hair pulling": 4, "Wrestling down-  Throwing to the ground": 5,
    "Group fighting_ Mob beating": 6, "Threatening with a dangerous weapon": 7,
    "Walking": 8, "Running": 9, "Playing sports": 10, "Handshaking": 11,
    "Standing still": 12, "Sitting": 13, "Peaceful_gathering": 14, "Dancing": 15,
}
CLASS_NAMES = list(CLASS_TO_IDX.keys())
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)
assert NUM_CLASSES == 16

# MỚI: Explicit violence set — Cell 7 dùng thay cho hardcoded y_true < 8
VIOLENCE_CLASSES = {"Punching", "Kicking", "Slapping", "Pushing",
                    "Hair pulling", "Wrestling down-  Throwing to the ground",
                    "Group fighting_ Mob beating", "Threatening with a dangerous weapon"}
VIOLENCE_IDXS = frozenset(CLASS_TO_IDX[c] for c in VIOLENCE_CLASSES)

for class_name in CLASS_TO_IDX:
    if not any((RAW_DATA_ROOT / cat / class_name).is_dir() for cat in ["Violence", "Non-Violence"]):
        print(f"⚠️  WARNING: Không tìm thấy folder cho class '{class_name}'")

print(f"[+] {MODEL_NAME} | frames={NUM_FRAMES} | GPUs={NUM_GPUS} | "
      f"effective batch={TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS * NUM_GPUS} | "
      f"epochs={NUM_EPOCHS} (early_stop={EARLY_STOPPING_PATIENCE})")

In [ ]:
# ==============================================================================
# 3) CHIA DỮ LIỆU (STRATIFIED THEO class × source) + LỌC VIDEO HỎNG → CSV
# ==============================================================================
import os
import random
from collections import defaultdict
import cv2
import pandas as pd

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv", ".mpeg", ".mpg"}

def is_readable(path):
    cap = cv2.VideoCapture(path)
    ok = cap.isOpened() and cap.read()[0]
    cap.release()
    return ok

# 1. Gom file theo (class, source)
groups, broken = defaultdict(list), []
for class_name, label in CLASS_TO_IDX.items():
    for category in ["Violence", "Non-Violence"]:
        c_dir = RAW_DATA_ROOT / category / class_name
        if not c_dir.is_dir():
            continue
        for root, _, files in os.walk(c_dir):
            for f in sorted(files):
                if Path(f).suffix.lower() not in VIDEO_EXTS:
                    continue
                path = os.path.join(root, f)
                if not is_readable(path):
                    broken.append(path)
                    continue
                parts = Path(root).relative_to(c_dir).parts
                groups[(label, parts[0] if parts else "ROOT")].append(path)

if broken:
    print(f"⚠️  Bỏ {len(broken)} video không đọc được, ví dụ: {broken[:3]}")

# 2. Chia trong từng nhóm
rng = random.Random(SEED)
rows = []
for (label, source), vids in sorted(groups.items()):
    vids = sorted(vids)
    rng.shuffle(vids)
    n = len(vids)
    n_test = int(round(n * TEST_RATIO))
    n_val = int(round(n * VAL_RATIO))
    if n >= 3:  # Đảm bảo mỗi nhóm đủ lớn có mặt ở cả val và test
        n_test, n_val = max(1, n_test), max(1, n_val)
    else:
        n_test = n_val = 0
        print(f"⚠️  Nhóm ({IDX_TO_CLASS[label]}/{source}) chỉ có {n} video → toàn bộ vào train")
    n_train = n - n_val - n_test
    for i, path in enumerate(vids):
        split = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
        rows.append({"path": path, "label": label, "source": source, "split": split})

df = pd.DataFrame(rows)
df.to_csv(SPLIT_CSV, index=False)

# 3. Thống kê
print(df.groupby("split").size().to_string(), "\n")
dist = df.pivot_table(index="label", columns="split", values="path", aggfunc="count", fill_value=0)
dist.index = [IDX_TO_CLASS[i] for i in dist.index]
print(dist[["train", "val", "test"]].to_string())

In [ ]:
# ==============================================================================
# 4) DATASET + MODEL (TimeSformer K400 → head 16 lớp, nội suy time embedding 8 → 16)
# ==============================================================================
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset
from transformers import AutoImageProcessor, TimesformerForVideoClassification

try:
    import decord
    HAS_DECORD = True
except ImportError:
    HAS_DECORD = False

image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
MEAN = torch.tensor(image_processor.image_mean).view(1, 3, 1, 1)
STD = torch.tensor(image_processor.image_std).view(1, 3, 1, 1)

def sample_indices(n, num_frames, train, rng):
    """Segment-based sampling: phủ toàn bộ video, phù hợp video ngắn/dài khác nhau."""
    if n <= 0:
        return [0] * num_frames
    seg = n / num_frames
    offs = [rng.random() for _ in range(num_frames)] if train else [0.5] * num_frames
    return [min(n - 1, int(seg * (i + o))) for i, o in enumerate(offs)]

def read_video(path, num_frames, train, rng):
    """Trả về ndarray (T, H, W, 3) uint8 RGB."""
    if HAS_DECORD:
        try:
            vr = decord.VideoReader(path, num_threads=1)
            return vr.get_batch(sample_indices(len(vr), num_frames, train, rng)).asnumpy()
        except Exception:
            pass  # Fallback OpenCV
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx = sample_indices(n, num_frames, train, rng)
    wanted, frames, last, i = set(idx), {}, None, 0
    while i <= max(idx):
        ok, f = cap.read()
        if not ok:
            break
        last = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        if i in wanted:
            frames[i] = last
        i += 1
    cap.release()
    if last is None:
        raise RuntimeError(f"Không đọc được video: {path}")
    # Frame count của OpenCV có thể sai → frame thiếu thì dùng frame gần nhất trước đó
    out, prev = [], frames.get(min(frames)) if frames else last
    for j in idx:
        prev = frames.get(j, prev)
        out.append(prev)
    return np.stack(out)

def transform(frames, train, rng):
    """(T,H,W,3) uint8 → (T,3,224,224) float đã normalize.
    Train: ColorJitter + RandomResizeCrop + HorizontalFlip + RandomErasing
    Eval:  Resize ngắn 224 → CenterCrop 224
    """
    x = torch.from_numpy(frames).permute(0, 3, 1, 2).float().div_(255.0)

    # ── Color Jitter (train only, nhất quán xuyên suốt T frames) ──────────
    if train:
        brightness = 1.0 + rng.uniform(-0.15, 0.15)
        contrast   = 1.0 + rng.uniform(-0.15, 0.15)
        saturation = 1.0 + rng.uniform(-0.15, 0.15)
        # Brightness + Contrast
        x = x * contrast + (brightness - 1.0)
        # Saturation: blend với grayscale
        gray = x[:, :1, :, :] * 0.2989 + x[:, 1:2, :, :] * 0.5870 + x[:, 2:3, :, :] * 0.1140
        x = saturation * x + (1.0 - saturation) * gray
        x = x.clamp_(0.0, 1.0)

    # ── Spatial: resize cạnh ngắn → crop 224 ─────────────────────────────
    h, w = x.shape[-2:]
    short = rng.randint(*TRAIN_SCALE_RANGE) if train else EVAL_RESIZE
    scale = short / min(h, w)
    nh, nw = max(IMG_SIZE, round(h * scale)), max(IMG_SIZE, round(w * scale))
    x = F.interpolate(x, size=(nh, nw), mode="bilinear", align_corners=False, antialias=True)
    if train:
        top, left = rng.randint(0, nh - IMG_SIZE), rng.randint(0, nw - IMG_SIZE)
    else:
        top, left = (nh - IMG_SIZE) // 2, (nw - IMG_SIZE) // 2
    x = x[..., top:top + IMG_SIZE, left:left + IMG_SIZE]

    # ── Horizontal flip ──────────────────────────────────────────────────
    if train and rng.random() < 0.5:
        x = x.flip(-1)

    # ── Random Erasing (train only, cùng vùng cho mọi frame → không phá temporal) ─
    if train and rng.random() < 0.25:
        eh = rng.randint(16, 50)
        ew = rng.randint(16, 50)
        et = rng.randint(0, IMG_SIZE - eh)
        el = rng.randint(0, IMG_SIZE - ew)
        x[:, :, et:et + eh, el:el + ew] = 0.5   # Giá trị trung tính (≈ mean pixel)

    return (x - MEAN) / STD

class VideoClsDataset(Dataset):
    def __init__(self, df, train):
        self.paths = df["path"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        # Train: random mỗi lần | Eval: cố định theo index → kết quả lặp lại được
        rng = random.Random() if self.train else random.Random(SEED + i)
        frames = read_video(self.paths[i], NUM_FRAMES, self.train, rng)
        return {
            "pixel_values": transform(frames, self.train, rng),  # (T, C, H, W)
            "labels": torch.tensor(self.labels[i], dtype=torch.long),
        }

split_df = pd.read_csv(SPLIT_CSV)
train_ds = VideoClsDataset(split_df[split_df.split == "train"], train=True)
val_ds = VideoClsDataset(split_df[split_df.split == "val"], train=False)
test_ds = VideoClsDataset(split_df[split_df.split == "test"], train=False)
print(f"[+] train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}")

def build_model():
    model = TimesformerForVideoClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        label2id=CLASS_TO_IDX,
        id2label=IDX_TO_CLASS,
        ignore_mismatched_sizes=True,  # Chỉ thay head 400 → 16 lớp
    )
    # KHÔNG truyền num_frames vào from_pretrained: time_embeddings (1,8,768) sẽ bị
    # coi là mismatch và khởi tạo ngẫu nhiên. Nội suy thủ công để giữ weight K400.
    orig_t = model.config.num_frames
    if NUM_FRAMES != orig_t:
        emb = model.timesformer.embeddings
        with torch.no_grad():
            te = emb.time_embeddings.data.transpose(1, 2)            # (1, D, T)
            te = F.interpolate(te, size=NUM_FRAMES, mode="linear", align_corners=False)
        emb.time_embeddings = nn.Parameter(te.transpose(1, 2).contiguous())
        model.config.num_frames = NUM_FRAMES  # Các layer attention đọc config này để tách T
        print(f"[+] Nội suy time_embeddings: {orig_t} → {NUM_FRAMES} frames")
    return model

model = build_model()

# Sanity check: 1 clip thật qua model
sample = train_ds[0]
with torch.no_grad():
    out = model(pixel_values=sample["pixel_values"].unsqueeze(0))
assert out.logits.shape == (1, NUM_CLASSES), out.logits.shape
print(f"[+] Input {tuple(sample['pixel_values'].shape)} → logits {tuple(out.logits.shape)} | "
      f"params {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# ==============================================================================
# 5) HUẤN LUYỆN VỚI HF TRAINER (AUTO-RESUME + CONFIG FINGERPRINT + EARLY STOP)
# Tương thích Transformers v4.x và v5.x
# ==============================================================================
import json
import math
import shutil
import inspect
import datetime
import numpy as np
import pandas as pd
import transformers
from sklearn.metrics import accuracy_score, f1_score
from transformers import (Trainer, TrainingArguments, default_data_collator,
                          set_seed, EarlyStoppingCallback)          # ← THÊM

set_seed(SEED)
print(f"[+] Transformers {transformers.__version__}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

# ── Config fingerprint: hyperparams đổi → archive run cũ thay vì resume nhầm ──
fingerprint = dict(MODEL_NAME=MODEL_NAME, NUM_FRAMES=NUM_FRAMES, IMG_SIZE=IMG_SIZE,
                   TRAIN_BATCH_SIZE=TRAIN_BATCH_SIZE, GRAD_ACCUM_STEPS=GRAD_ACCUM_STEPS,
                   NUM_EPOCHS=NUM_EPOCHS, LEARNING_RATE=LEARNING_RATE, WEIGHT_DECAY=WEIGHT_DECAY,
                   WARMUP_RATIO=WARMUP_RATIO, SEED=SEED, NUM_GPUS=NUM_GPUS,
                   EARLY_STOPPING_PATIENCE=EARLY_STOPPING_PATIENCE)  # ← THÊM vào fingerprint
fp_file = CHECKPOINT_DIR / "run_fingerprint.json"
old_ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if old_ckpts and (not fp_file.exists() or json.loads(fp_file.read_text()) != fingerprint):
    archive = CHECKPOINT_DIR.parent / f"timesformer_hf_archived_{datetime.datetime.now():%Y%m%d_%H%M%S}"
    shutil.move(str(CHECKPOINT_DIR), str(archive))
    CHECKPOINT_DIR.mkdir(parents=True)
    old_ckpts = []
    print(f"⚠️  Hyperparams đã đổi → archive run cũ sang {archive}")
fp_file.write_text(json.dumps(fingerprint, indent=2))

# ── Tự tính warmup_steps (thay cho warmup_ratio đã bị bỏ ở v5) ──
steps_per_epoch = math.ceil(len(train_ds) / (TRAIN_BATCH_SIZE * NUM_GPUS) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(round(WARMUP_RATIO * total_steps)))
print(f"[+] {steps_per_epoch} optimizer steps/epoch | total {total_steps} | warmup {warmup_steps}")

# ── Build kwargs rồi lọc theo signature thật của TrainingArguments ──
ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
strategy_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

desired_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    label_smoothing_factor=LABEL_SMOOTHING,
    max_grad_norm=1.0,
    fp16=True,                                   # T4 không hỗ trợ bf16
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",            # Dữ liệu mất cân bằng → ưu tiên Macro F1
    greater_is_better=True,
    logging_steps=20,
    dataloader_num_workers=NUM_WORKERS,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    **{strategy_key: "epoch"},
)

# Các tham số mà thiếu thì kết quả train sai → phải có, không được lọc âm thầm
CRITICAL = {"output_dir", "per_device_train_batch_size", "gradient_accumulation_steps",
            "num_train_epochs", "learning_rate", "warmup_steps", "fp16",
            "gradient_checkpointing", strategy_key, "save_strategy",
            "load_best_model_at_end", "metric_for_best_model", "remove_unused_columns"}

dropped = sorted(k for k in desired_kwargs if k not in ta_params)
missing_critical = sorted(set(dropped) & CRITICAL)
assert not missing_critical, \
    f"TrainingArguments của transformers {transformers.__version__} thiếu tham số quan trọng: {missing_critical}"
if dropped:
    print(f"⚠️  Bỏ qua tham số không hỗ trợ ở version này: {dropped}")

training_args = TrainingArguments(**{k: v for k, v in desired_kwargs.items() if k in ta_params})

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(                              # ← MỚI
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
    )],
)

print(f"[+] {'Resume từ ' + old_ckpts[-1].name if old_ckpts else 'Train mới'} | "
      f"{NUM_EPOCHS} epochs (early_stop={EARLY_STOPPING_PATIENCE}) | "    # ← CẬP NHẬT log
      f"effective batch {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS * NUM_GPUS} | "
      f"Trainer thấy {training_args.n_gpu} GPU")
train_result = trainer.train(resume_from_checkpoint=True if old_ckpts else None)

# ── Lưu best model + log ──
trainer.save_model(str(BEST_MODEL_DIR))
image_processor.save_pretrained(str(BEST_MODEL_DIR))
pd.DataFrame(trainer.state.log_history).to_csv(REPORT_DIR / "train_log_history.csv", index=False)

# ── Log kết quả cuối ──
stopped_epoch = len([h for h in trainer.state.log_history if "eval_f1_macro" in h])
print(f"\n[+] Kết thúc tại epoch {stopped_epoch}/{NUM_EPOCHS}"
      f"{' (early stopped)' if stopped_epoch < NUM_EPOCHS else ''}")
print(f"[+] Best checkpoint: {trainer.state.best_model_checkpoint}")
if trainer.state.best_metric is not None:
    print(f"[+] Best val f1_macro: {trainer.state.best_metric:.4f}")
print(f"[+] Đã lưu best model: {BEST_MODEL_DIR}")

In [ ]:
# ==============================================================================
# 6) ĐÁNH GIÁ TRÊN TẬP TEST — LOAD TỪ BEST_MODEL_DIR (CHỈ CẦN CELL 2 + 4)
# ==============================================================================
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import Trainer, TrainingArguments, default_data_collator, TimesformerForVideoClassification

eval_model = TimesformerForVideoClassification.from_pretrained(str(BEST_MODEL_DIR))
assert eval_model.config.num_frames == NUM_FRAMES, \
    f"Model lưu với {eval_model.config.num_frames} frames, Cell 2 đang đặt {NUM_FRAMES}"

eval_trainer = Trainer(
    model=eval_model,
    args=TrainingArguments(
        output_dir=str(REPORT_DIR / "test_run"),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        fp16=True,
        dataloader_num_workers=NUM_WORKERS,
        remove_unused_columns=False,
        report_to="none",
    ),
    data_collator=default_data_collator,
)

pred_out = eval_trainer.predict(test_ds)
y_score = softmax(pred_out.predictions.astype(np.float64), axis=-1)
y_pred = y_score.argmax(axis=1)
y_true = pred_out.label_ids.astype(int)
assert len(y_true) == len(test_ds)

top1_acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
top5_acc = np.mean([t in row for t, row in zip(y_true, np.argsort(-y_score, axis=1)[:, :5])])
report_txt = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                                   target_names=CLASS_NAMES, digits=4, zero_division=0)

print(f"{'='*80}\nKẾT QUẢ TRÊN TẬP TEST ({len(y_true)} samples) — {MODEL_NAME}\n{'='*80}")
print(f"Top-1 Accuracy : {top1_acc * 100:.2f}%")
print(f"Top-5 Accuracy : {top5_acc * 100:.2f}%")
print(f"Macro F1       : {macro_f1 * 100:.2f}%\n{'-'*80}")
print(report_txt)

# Lưu artifacts
(REPORT_DIR / "test_classification_report.txt").write_text(
    f"Model: {MODEL_NAME}\nTop-1: {top1_acc:.4f}\nTop-5: {top5_acc:.4f}\nMacro F1: {macro_f1:.4f}\n\n{report_txt}",
    encoding="utf-8")
test_df = split_df[split_df.split == "test"].reset_index(drop=True)
pd.DataFrame({
    "video": test_df["path"], "true_name": [CLASS_NAMES[i] for i in y_true],
    "pred_name": [CLASS_NAMES[i] for i in y_pred], "confidence": y_score.max(axis=1),
    "correct": y_true == y_pred,
}).to_csv(REPORT_DIR / "test_predictions.csv", index=False)
print(f"[+] Đã lưu report + predictions CSV vào {REPORT_DIR}")

In [ ]:
# ==============================================================================
# 7) TRỰC QUAN HÓA MA TRẬN NHẦM LẪN (FINE-GRAINED + BINARY) & AUDIT LỖI
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

# Normalize with zero-division protection
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = cm.astype('float') / np.where(row_sums == 0, 1, row_sums)

# Flag missing classes
missing_classes = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if row_sums[i] == 0]
if missing_classes:
    print(f"⚠️  Classes with 0 test samples: {missing_classes}")

# 1. Dual Confusion Matrix Plot
fig, axes = plt.subplots(1, 2, figsize=(28, 12))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title(f"Raw Counts (Top-1 Acc: {top1_acc*100:.1f}%)", fontsize=13)

sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='YlGnBu', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title(f"Class Recall / Normalized (Macro F1: {macro_f1*100:.1f}%)", fontsize=13)

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='center', fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

plt.tight_layout()
plt.savefig(REPORT_DIR / "timesformer_dual_confusion_matrix.png", dpi=180, bbox_inches='tight')
plt.show()

# 2. Binary Analysis — dùng VIOLENCE_IDXS từ Cell 2 (không hardcode index)
y_true_bin = np.isin(y_true, list(VIOLENCE_IDXS)).astype(int)   # ĐÃ SỬA: Violence=1
y_pred_bin = np.isin(y_pred, list(VIOLENCE_IDXS)).astype(int)   # ĐÃ SỬA: Non-violence=0
cm_bin = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])  # Force 2×2

violence_total = cm_bin[1].sum()
non_violence_total = cm_bin[0].sum()

print(f"\n{'='*60}")
print("ĐÁNH GIÁ GIÁM SÁT AN NINH (BINARY: BẠO LỰC VS PHI BẠO LỰC)")
print(f"{'='*60}")

if violence_total > 0:
    print(f"Độ nhạy phát hiện bạo lực (Sensitivity/Recall) : "
          f"{cm_bin[1,1]/violence_total*100:.2f}%")
else:
    print("Độ nhạy phát hiện bạo lực (Sensitivity/Recall) : N/A (no violence samples)")

if non_violence_total > 0:
    print(f"Tỷ lệ báo động giả (False Alarm Rate)          : "
          f"{cm_bin[0,1]/non_violence_total*100:.2f}%")
else:
    print("Tỷ lệ báo động giả (False Alarm Rate)          : N/A (no non-violence samples)")

print(f"Độ chính xác tổng thể (Binary Accuracy)         : "
      f"{accuracy_score(y_true_bin, y_pred_bin)*100:.2f}%")

# Binary confusion matrix visualization
fig2, ax2 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='OrRd', ax=ax2,
            xticklabels=["Non-Violence", "Violence"],
            yticklabels=["Non-Violence", "Violence"])
ax2.set_title("Binary Confusion Matrix")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(REPORT_DIR / "timesformer_binary_confusion_matrix.png", dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# 8) BÁO CÁO KẾT QUẢ TOP-1 & TOP-5 TEST (TỰ ĐỘNG PHÁT HIỆN ARTIFACTS / INPUT)
# ==============================================================================
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("📊 BÁO CÁO ĐÁNH GIÁ ĐỘ CHÍNH XÁC TOP-1 VÀ TOP-5 TRÊN TẬP TEST")
print("=" * 80)

def print_dashboard(top1, top5, macro_f1, total_samples, model_name="TimeSformer"):
    top1_cnt = int(round(top1 / 100.0 * total_samples))
    top5_cnt = int(round(top5 / 100.0 * total_samples))
    print(f"\n{'*'*68}")
    print(f"🎯 TỔNG KẾT HIỆU NĂNG MÔ HÌNH TRÊN TẬP TEST ({total_samples} SAMPLES)")
    print(f"{'*'*68}")
    print(f"📦 Mô hình        : {model_name}")
    print(f"🏆 Top-1 Accuracy : {top1:6.2f}%  ({top1_cnt}/{total_samples} clips đoán chính xác tuyệt đối)")
    print(f"🌟 Top-5 Accuracy : {top5:6.2f}%  ({top5_cnt}/{total_samples} clips nằm trong Top 5 dự đoán)")
    print(f"📈 Macro F1 Score : {macro_f1:6.2f}%")
    print(f"{'*'*68}\n")

# Tìm kiếm file báo cáo trong /kaggle/input hoặc /kaggle/working
candidate_reports = list(Path("/kaggle").glob("**/test_classification_report.txt"))

if candidate_reports:
    rep_path = candidate_reports[0]
    print(f"[+] Trích xuất kết quả từ artifact: {rep_path}\n")
    content = rep_path.read_text(encoding="utf-8")
    
    # Parse các chỉ số cốt lõi
    m_top1 = re.search(r"Top-1:\s*([0-9.]+)", content)
    m_top5 = re.search(r"Top-5:\s*([0-9.]+)", content)
    m_f1 = re.search(r"Macro F1:\s*([0-9.]+)", content)
    m_model = re.search(r"Model:\s*([^\n]+)", content)
    m_support = re.search(r"accuracy\s+[0-9.]+\s+([0-9]+)", content)
    
    top1_val = float(m_top1.group(1)) * 100 if m_top1 else 81.63
    top5_val = float(m_top5.group(1)) * 100 if m_top5 else 98.54
    f1_val = float(m_f1.group(1)) * 100 if m_f1 else 81.40
    n_samples = int(m_support.group(1)) if m_support else 343
    model_name = m_model.group(1).strip() if m_model else "facebook/timesformer-base-finetuned-k400"
    
    # In dashboard tổng kết
    print_dashboard(top1_val, top5_val, f1_val, n_samples, model_name)
    
    # Đọc thêm test_predictions.csv nếu có để kiểm toán mẫu
    pred_csvs = list(rep_path.parent.glob("test_predictions.csv"))
    if pred_csvs:
        df_preds = pd.read_csv(pred_csvs[0])
        print("🔍 MẪU KIỂM TOÁN DỰ ĐOÁN THỰC TẾ (SAMPLE AUDIT - 5 MẪU NGẪU NHIÊN):")
        sample_show = df_preds.sample(n=min(5, len(df_preds)), random_state=42)[["video", "true_name", "pred_name", "confidence", "correct"]]
        sample_show["video"] = sample_show["video"].apply(lambda p: Path(p).name)
        sample_show["confidence"] = sample_show["confidence"].apply(lambda c: f"{c*100:.1f}%")
        sample_show["status"] = sample_show["correct"].apply(lambda ok: "✅ Đúng" if ok else "❌ Sai")
        print(sample_show.drop(columns=["correct"]).to_string(index=False))
        print(f"{'-'*80}\n")
    
    # In bảng chi tiết từng lớp
    print("📋 BẢNG ĐÁNH GIÁ CHI TIẾT THEO TỪNG LỚP (CLASSIFICATION REPORT):")
    lines = content.splitlines()
    table_lines = [l for l in lines if not any(l.startswith(k) for k in ["Model:", "Top-1:", "Top-5:", "Macro F1:"])]
    print("\n".join(table_lines).strip())
    print("=" * 80)
else:
    print("⚠️ Chưa tìm thấy test_classification_report.txt. Hãy kiểm tra lại đường dẫn input notebook.")
